# GKR Demo (Pytorch version)

This notebook keeps the original demo workflow, but it is fully implemented by pytorch

## Difference from the original demo: covariance-kernel initialization

The [original GKR demo](https://github.com/AgeYY/speed_grid_cell_information/blob/main/GKR_demo_torch_colab.ipynb) initializes the lower-triangular covariance-kernel precision factor from random normal values. This notebook instead uses a data-informed starting kernel size after fitting the GP mean, while leaving the subsequent covariance optimization unchanged.

Let $\lambda_j$ be the eigenvalues of the sample covariance of the mean-model residuals. Their participation ratio gives the residual effective dimensionality:

$$
d_{\mathrm{eff}} = \frac{(\sum_j \lambda_j)^2}{\sum_j \lambda_j^2}.
$$

For a one-dimensional periodic condition, the covariance estimator uses weights

$$
w_i(\theta; p) = \exp\left[-p\,\sin^4\left(\frac{\pi(\theta_i-\theta)}{T}\right)\right],
$$

where $T$ is the period and $p$ is the kernel precision. The effective sample size at a query is

$$
N_{\mathrm{eff}}(\theta; p) = \frac{(\sum_i w_i)^2}{\sum_i w_i^2}.
$$

The initialization targets $N_{\mathrm{eff,target}} = \operatorname{clip}(k d_{\mathrm{eff}}, 1, n)$, with $k=10$ in this demo. Bisection chooses $p$ so the mean effective sample size over a periodic grid matches this target, then initializes the precision factor with $L_{00}=\sqrt{p}$, so $LL^\top=p$ in the one-dimensional case. The learned covariance kernel is still optimized afterward with the same validation Gaussian log-likelihood objective. Setting `covariance_neighbors_per_effective_dimension=None` retains the original random initialization.

In [ ]:
# Colab / notebook dependency setup
%pip install -q --upgrade pip
%pip install -q numpy matplotlib scikit-learn gpytorch

In [ ]:
import importlib
import subprocess
import sys

if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])

print('Torch is available.')

## Inlined implementation

Model

In [ ]:
import math
from pathlib import Path

import gpytorch
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Ellipse

def plot_cov_ellipse(mean, cov, ax, n_std=2.0, **kwargs):
    eigenvals, eigenvecs = np.linalg.eigh(cov)
    order = np.argsort(eigenvals)[::-1]
    eigenvals = eigenvals[order]
    eigenvecs = eigenvecs[:, order]
    angle = np.degrees(np.arctan2(eigenvecs[1, 0], eigenvecs[0, 0]))
    width, height = 2 * n_std * np.sqrt(np.maximum(eigenvals, 0.0))
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, fill=True, **kwargs)
    ax.add_patch(ellipse)
    return ellipse

print("Torch version:", torch.__version__)
print("Using device:", "cuda" if torch.cuda.is_available() else "cpu")

import copy
import math
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from typing import Optional

import gpytorch
import numpy as np
import torch
from sklearn.covariance import GraphicalLasso


TensorLike = np.ndarray | torch.Tensor


def residual_participation_ratio(residuals: TensorLike) -> float:
    """Return the effective dimensionality of the residual covariance."""

    values = np.asarray(
        torch.as_tensor(residuals).detach().cpu(),
        dtype=np.float64,
    )
    if values.ndim != 2 or values.shape[0] < 2 or values.shape[1] < 1:
        raise ValueError("residuals must have shape [n >= 2, d >= 1]")
    centered = values - values.mean(axis=0, keepdims=True)
    covariance = centered.T @ centered / float(values.shape[0] - 1)
    eigenvalues = np.maximum(np.linalg.eigvalsh(covariance), 0.0)
    denominator = float(np.square(eigenvalues).sum())
    if denominator <= 0.0:
        return 1.0
    return float(np.square(eigenvalues.sum()) / denominator)


def periodic_covariance_kernel_initialization(
    residuals: TensorLike,
    conditions: TensorLike,
    *,
    period: float,
    neighbors_per_effective_dimension: float = 5.0,
    grid_size: int = 256,
) -> dict[str, float]:
    """Choose kernel size from residual dimensionality and neighborhood ESS.

    The covariance estimator below uses weights proportional to
    exp(-precision * sin(pi * delta / period)**4). The target effective sample
    size is the residual participation ratio times
    neighbors_per_effective_dimension.
    """

    theta = np.asarray(
        torch.as_tensor(conditions).detach().cpu(),
        dtype=np.float64,
    ).reshape(-1)
    if theta.size != np.asarray(torch.as_tensor(residuals).detach().cpu()).shape[0]:
        raise ValueError("conditions and residuals must contain the same rows")
    if not np.isfinite(period) or period <= 0.0:
        raise ValueError("period must be positive and finite")
    if (
        not np.isfinite(neighbors_per_effective_dimension)
        or neighbors_per_effective_dimension <= 0.0
    ):
        raise ValueError("neighbors_per_effective_dimension must be positive")
    if int(grid_size) < 8:
        raise ValueError("grid_size must be at least 8")
    participation = residual_participation_ratio(residuals)
    target = float(
        np.clip(
            neighbors_per_effective_dimension * participation,
            1.0,
            float(theta.size),
        )
    )
    grid = np.linspace(
        theta.min(),
        theta.min() + float(period),
        int(grid_size),
        endpoint=False,
    )
    geometry = np.sin(
        np.pi * (theta[:, None] - grid[None, :]) / float(period)
    ) ** 4

    def effective_size(log_precision: float) -> float:
        weights = np.exp(-math.exp(log_precision) * geometry)
        numerator = np.square(weights.sum(axis=0))
        denominator = np.square(weights).sum(axis=0).clip(min=1e-300)
        return float(np.mean(numerator / denominator))

    lower = math.log(1e-8)
    upper = math.log(1e8)
    for _ in range(80):
        midpoint = 0.5 * (lower + upper)
        if effective_size(midpoint) > target:
            lower = midpoint
        else:
            upper = midpoint

    initial_precision = math.exp(0.5 * (lower + upper))
    return {
        "residual_participation_ratio": participation,
        "neighbors_per_effective_dimension": float(
            neighbors_per_effective_dimension
        ),
        "target_effective_sample_size": target,
        "initial_effective_sample_size": effective_size(
            math.log(initial_precision)
        ),
        "initial_kernel_size": 1.0 / initial_precision,
        "initial_precision": initial_precision,
    }


def _dtype_to_name(dtype: torch.dtype) -> str:
    return str(dtype).replace("torch.", "")


def _dtype_from_name(name: str) -> torch.dtype:
    if not hasattr(torch, name):
        raise ValueError(f"Unsupported torch dtype name: {name}")
    dtype = getattr(torch, name)
    if not isinstance(dtype, torch.dtype):
        raise ValueError(f"Resolved object is not a torch dtype: {name}")
    return dtype


def _ensure_2d_tensor(
    value: TensorLike,
    *,
    dtype: torch.dtype,
    device: torch.device,
) -> torch.Tensor:
    tensor = torch.as_tensor(value, dtype=dtype, device=device)
    if tensor.ndim == 1:
        tensor = tensor.unsqueeze(-1)
    if tensor.ndim != 2:
        raise ValueError(f"Expected a 2D array, got shape {tuple(tensor.shape)}.")
    return tensor


def _circular_diff(diff: torch.Tensor, periods: Optional[float | list[Optional[float]]]) -> torch.Tensor:
    if periods is None:
        return diff

    if np.isscalar(periods):
        period = float(periods)
        return torch.sin(torch.pi * diff / period).pow(2)

    diff = diff.clone()
    for dim, period in enumerate(periods):
        if period is not None:
            diff[..., dim] = torch.sin(torch.pi * diff[..., dim] / float(period)).pow(2)
    return diff


def gaussian_log_likelihood(
    responses: torch.Tensor,
    covariance: torch.Tensor,
    *,
    diag_factor: float = 1e-5,
) -> torch.Tensor:
    responses = torch.as_tensor(responses, dtype=covariance.dtype, device=covariance.device)
    eye = torch.eye(covariance.shape[-1], dtype=covariance.dtype, device=covariance.device)
    stabilized = covariance + eye.unsqueeze(0) * diag_factor
    chol = torch.linalg.cholesky(stabilized)
    log_det = 2.0 * torch.log(torch.diagonal(chol, dim1=-2, dim2=-1)).sum(dim=-1)
    solved = torch.linalg.solve_triangular(chol, responses.unsqueeze(-1), upper=False)
    quadratic = solved.square().sum(dim=(-2, -1))
    return -0.5 * torch.mean(log_det + quadratic)


def _build_dimension_kernel(
    period: Optional[float],
    active_dim: int,
    *,
    batch_shape: Optional[torch.Size] = None,
) -> gpytorch.kernels.Kernel:
    kwargs = {"active_dims": (active_dim,)}
    if batch_shape is not None:
        kwargs["batch_shape"] = batch_shape

    if period is None:
        return gpytorch.kernels.RBFKernel(**kwargs)

    kernel = gpytorch.kernels.PeriodicKernel(**kwargs)
    kernel.period_length = float(period)
    return kernel


def create_kernel(
    *,
    circular_period: Optional[float | list[Optional[float]]],
    input_dim: int,
    batch_shape: Optional[torch.Size] = None,
) -> gpytorch.kernels.Kernel:
    if circular_period is None:
        kwargs = {"ard_num_dims": input_dim}
        if batch_shape is not None:
            kwargs["batch_shape"] = batch_shape
        base = gpytorch.kernels.RBFKernel(**kwargs)
        if batch_shape is None:
            return gpytorch.kernels.ScaleKernel(base)
        return gpytorch.kernels.ScaleKernel(base, batch_shape=batch_shape)

    if np.isscalar(circular_period):
        kernels = [
            _build_dimension_kernel(float(circular_period), active_dim, batch_shape=batch_shape)
            for active_dim in range(input_dim)
        ]
    elif isinstance(circular_period, list):
        if len(circular_period) != input_dim:
            raise ValueError("Length of circular_period must match input_dim.")
        kernels = [
            _build_dimension_kernel(period, active_dim, batch_shape=batch_shape)
            for active_dim, period in enumerate(circular_period)
        ]
    else:
        raise ValueError("Invalid input for circular_period.")

    combined = kernels[0]
    for kernel in kernels[1:]:
        combined = combined * kernel
    if batch_shape is None:
        return gpytorch.kernels.ScaleKernel(combined)
    return gpytorch.kernels.ScaleKernel(combined, batch_shape=batch_shape)


class _ExactGPModel(gpytorch.models.ExactGP):
    def __init__(
        self,
        train_x: torch.Tensor,
        train_y: torch.Tensor,
        likelihood: gpytorch.likelihoods.GaussianLikelihood,
        kernel: gpytorch.kernels.Kernel,
    ) -> None:
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = kernel

    def forward(self, x: torch.Tensor) -> gpytorch.distributions.MultivariateNormal:
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


class _SparseGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, inducing_points: torch.Tensor, kernel: gpytorch.kernels.Kernel) -> None:
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            inducing_points.size(0)
        )
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True,
        )
        super().__init__(variational_strategy)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = kernel

    def forward(self, x: torch.Tensor) -> gpytorch.distributions.MultivariateNormal:
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


class _BatchExactGPModel(gpytorch.models.ExactGP):
    def __init__(
        self,
        train_x: torch.Tensor,
        train_y: torch.Tensor,
        likelihood: gpytorch.likelihoods.GaussianLikelihood,
        kernel: gpytorch.kernels.Kernel,
        batch_shape: torch.Size,
    ) -> None:
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean(batch_shape=batch_shape)
        self.covar_module = kernel

    def forward(self, x: torch.Tensor) -> gpytorch.distributions.MultivariateNormal:
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


class _BatchSparseGPModel(gpytorch.models.ApproximateGP):
    def __init__(
        self,
        inducing_points: torch.Tensor,
        kernel: gpytorch.kernels.Kernel,
        batch_shape: torch.Size,
    ) -> None:
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            inducing_points.size(-2),
            batch_shape=batch_shape,
        )
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self,
            inducing_points,
            variational_distribution,
            learn_inducing_locations=True,
        )
        super().__init__(variational_strategy)
        self.mean_module = gpytorch.means.ConstantMean(batch_shape=batch_shape)
        self.covar_module = kernel

    def forward(self, x: torch.Tensor) -> gpytorch.distributions.MultivariateNormal:
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


@dataclass
class IndependentExactGPRegressor:
    n_outputs: int
    circular_period: Optional[float | list[Optional[float]]] = None
    standardize: bool = True
    n_inducing: Optional[int] = None
    seperate_kernel: bool = False
    training_iter: int = 1500
    lr: float = 0.1
    verbose: bool = True
    verbose_every: int = 10
    track_history: bool = False
    kernel: Optional[gpytorch.kernels.Kernel] = None
    dtype: torch.dtype = torch.float64
    device: Optional[str | torch.device] = None

    def __post_init__(self) -> None:
        self.device = torch.device(
            self.device or ("cuda" if torch.cuda.is_available() else "cpu")
        )
        if self.verbose_every < 1:
            raise ValueError("verbose_every must be >= 1.")
        self.output_mean_: Optional[torch.Tensor] = None
        self.output_std_: Optional[torch.Tensor] = None
        self.batch_model_: Optional[gpytorch.models.GP] = None
        self.batch_likelihood_: Optional[gpytorch.likelihoods.GaussianLikelihood] = None
        self.models_: list[gpytorch.models.GP] = []
        self.likelihoods_: list[gpytorch.likelihoods.GaussianLikelihood] = []
        self.mean_loss_history_: list[list[float]] = []

    @staticmethod
    def _mean_lengthscale(model: gpytorch.models.GP) -> Optional[float]:
        kernel = getattr(model, "covar_module", None)
        if kernel is None:
            return None

        base_kernel = getattr(kernel, "base_kernel", kernel)
        lengthscale = getattr(base_kernel, "lengthscale", None)
        if lengthscale is None:
            return None

        values = lengthscale.detach().reshape(-1)
        if values.numel() == 0:
            return None
        return float(values.mean().cpu())

    def _kernel_for_output(self, input_dim: int) -> gpytorch.kernels.Kernel:
        if self.kernel is None:
            return create_kernel(circular_period=self.circular_period, input_dim=input_dim)
        return copy.deepcopy(self.kernel)

    def _kernel_for_batch(self, input_dim: int) -> gpytorch.kernels.Kernel:
        batch_shape = torch.Size([self.n_outputs])
        if self.kernel is None:
            return create_kernel(
                circular_period=self.circular_period,
                input_dim=input_dim,
                batch_shape=batch_shape,
            )
        return copy.deepcopy(self.kernel)

    def _select_inducing_inputs(self, train_x: torch.Tensor, n_inducing: int) -> torch.Tensor:
        if n_inducing >= train_x.shape[0]:
            return train_x.clone()
        indices = torch.linspace(
            0,
            train_x.shape[0] - 1,
            steps=n_inducing,
            device=train_x.device,
        ).round().long()
        return train_x.index_select(0, indices)

    def fit(self, train_x: TensorLike, train_y: TensorLike) -> None:
        train_x = _ensure_2d_tensor(train_x, dtype=self.dtype, device=self.device)
        train_y = _ensure_2d_tensor(train_y, dtype=self.dtype, device=self.device)

        if train_y.shape[1] != self.n_outputs:
            raise ValueError("Number of output dimensions does not match n_outputs.")

        if self.n_inducing is not None:
            if not isinstance(self.n_inducing, int):
                raise ValueError("n_inducing must be an integer when provided.")
            if self.n_inducing < 1:
                raise ValueError("n_inducing must be a positive integer when provided.")

        if self.standardize:
            self.output_mean_ = train_y.mean(dim=0)
            self.output_std_ = train_y.std(dim=0).clamp_min(1e-8)
            normalized_y = (train_y - self.output_mean_) / self.output_std_
        else:
            self.output_mean_ = torch.zeros(train_y.shape[1], dtype=self.dtype, device=self.device)
            self.output_std_ = torch.ones(train_y.shape[1], dtype=self.dtype, device=self.device)
            normalized_y = train_y

        self.models_ = []
        self.likelihoods_ = []
        self.mean_loss_history_ = []
        self.batch_model_ = None
        self.batch_likelihood_ = None

        batch_shape = torch.Size([self.n_outputs])
        likelihood = gpytorch.likelihoods.GaussianLikelihood(batch_shape=batch_shape).to(self.device, self.dtype)
        kernel = self._kernel_for_batch(train_x.shape[1]).to(self.device, self.dtype)
        train_targets = normalized_y.transpose(0, 1).contiguous()

        if self.n_inducing is None:
            model: gpytorch.models.GP = _BatchExactGPModel(
                train_x=train_x,
                train_y=train_targets,
                likelihood=likelihood,
                kernel=kernel,
                batch_shape=batch_shape,
            ).to(self.device, self.dtype)
        else:
            n_inducing = min(int(self.n_inducing), train_x.shape[0])
            inducing_inputs = self._select_inducing_inputs(train_x, n_inducing)
            inducing_points = inducing_inputs.unsqueeze(0).repeat(self.n_outputs, 1, 1)
            model = _BatchSparseGPModel(
                inducing_points=inducing_points,
                kernel=kernel,
                batch_shape=batch_shape,
            ).to(self.device, self.dtype)

        model.train()
        likelihood.train()

        optimizer = torch.optim.Adam(model.parameters(), lr=self.lr)
        if self.n_inducing is None:
            mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
        else:
            mll = gpytorch.mlls.VariationalELBO(
                likelihood,
                model,
                num_data=train_x.shape[0],
            )

        if self.track_history:
            self.mean_loss_history_ = [[] for _ in range(self.n_outputs)]

        for iter_idx in range(self.training_iter):
            optimizer.zero_grad()
            output = model(train_x)
            loss_per_output = -mll(output, train_targets)
            if loss_per_output.ndim == 0:
                loss = loss_per_output
                detached_loss = loss_per_output.detach().repeat(self.n_outputs)
            else:
                loss = loss_per_output.sum()
                detached_loss = loss_per_output.detach().reshape(-1)
            loss.backward()
            optimizer.step()

            if self.track_history:
                for output_idx in range(self.n_outputs):
                    self.mean_loss_history_[output_idx].append(float(detached_loss[output_idx].cpu()))

            if self.verbose and (
                iter_idx == 0
                or (iter_idx + 1) % self.verbose_every == 0
                or (iter_idx + 1) == self.training_iter
            ):
                noise_value = float(likelihood.noise.detach().mean().cpu())
                lengthscale_value = self._mean_lengthscale(model)
                if lengthscale_value is None:
                    lengthscale_msg = "n/a"
                else:
                    lengthscale_msg = f"{lengthscale_value:.6f}"
                loss_msg = float(detached_loss.mean().cpu())
                print(
                    "[GPR] "
                    f"output=all/{self.n_outputs} "
                    f"iter={iter_idx + 1}/{self.training_iter} "
                    f"mean_loss={loss_msg:.6f} "
                    f"noise={noise_value:.6f} "
                    f"lengthscale_mean={lengthscale_msg}"
                )

        self.batch_model_ = model
        self.batch_likelihood_ = likelihood
        self.models_ = [model]
        self.likelihoods_ = [likelihood]

    def predict(self, query: TensorLike) -> tuple[torch.Tensor, torch.Tensor]:
        query = _ensure_2d_tensor(query, dtype=self.dtype, device=self.device)
        if self.batch_model_ is None or self.batch_likelihood_ is None:
            raise RuntimeError("Mean model must be fit before predict is called.")

        self.batch_model_.eval()
        self.batch_likelihood_.eval()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=gpytorch.utils.warnings.GPInputWarning)
            with torch.no_grad(), gpytorch.settings.fast_pred_var():
                posterior = self.batch_model_(query)
        mean = posterior.mean.transpose(0, 1).contiguous()
        variance = posterior.variance.transpose(0, 1).contiguous()
        mean = mean * self.output_std_ + self.output_mean_
        variance = variance * self.output_std_.pow(2)
        return mean, variance


class TorchKernelCovariance(torch.nn.Module):
    def __init__(
        self,
        n_input: int,
        n_output: int,
        circular_period: Optional[float | list[Optional[float]]] = None,
        *,
        diag_factor: float = 1e-6,
        dtype: torch.dtype = torch.float64,
        device: Optional[str | torch.device] = None,
    ) -> None:
        super().__init__()
        self.n_input = n_input
        self.n_output = n_output
        self.circular_period = circular_period
        self.diag_factor = diag_factor
        self.dtype = dtype
        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )

        initial = torch.randn(n_input, n_input, dtype=dtype, device=self.device)
        self.kernel_prec_L = torch.nn.Parameter(torch.tril(initial))

        self.register_buffer("train_responses", torch.empty(0, n_output, dtype=dtype, device=self.device))
        self.register_buffer("train_inputs", torch.empty(0, n_input, dtype=dtype, device=self.device))

    def fit(self, responses: TensorLike, inputs: TensorLike) -> None:
        responses = _ensure_2d_tensor(responses, dtype=self.dtype, device=self.device)
        inputs = _ensure_2d_tensor(inputs, dtype=self.dtype, device=self.device)
        self.train_responses = responses
        self.train_inputs = inputs

    def _precision_matrix(self) -> torch.Tensor:
        lower = torch.tril(self.kernel_prec_L)
        return lower @ lower.transpose(-1, -2)

    def predict_cov(self, query: TensorLike, pred_batch_size: int = 1000) -> torch.Tensor:
        if self.train_inputs.numel() == 0:
            raise RuntimeError("Kernel covariance estimator must be fit before predict_cov is called.")

        query = _ensure_2d_tensor(query, dtype=self.dtype, device=self.device)
        precision = self._precision_matrix()
        n_query = query.shape[0]

        cov_pred = torch.zeros(n_query, self.n_output, self.n_output, dtype=self.dtype, device=self.device)
        kernel_sum_total = torch.zeros(n_query, dtype=self.dtype, device=self.device)

        for start in range(0, self.train_inputs.shape[0], pred_batch_size):
            end = min(start + pred_batch_size, self.train_inputs.shape[0])
            batch_inputs = self.train_inputs[start:end]
            batch_responses = self.train_responses[start:end]

            diff = batch_inputs.unsqueeze(1) - query.unsqueeze(0)
            diff = _circular_diff(diff, self.circular_period)

            diff_prec = torch.einsum("bqi,ij,bqj->bq", diff, precision, diff)
            kernel_matrix = torch.exp(-diff_prec)
            kernel_sum_total = kernel_sum_total + kernel_matrix.sum(dim=0)

            gram = torch.einsum("bi,bj->bij", batch_responses, batch_responses)
            cov_pred = cov_pred + torch.einsum("bq,bij->qij", kernel_matrix, gram)

        cov_pred = cov_pred / kernel_sum_total.clamp_min(1e-12).view(-1, 1, 1)
        eye = torch.eye(self.n_output, dtype=self.dtype, device=self.device)
        return cov_pred + eye.unsqueeze(0) * self.diag_factor

    def forward(self, query: TensorLike, pred_batch_size: int = 1000) -> torch.Tensor:
        return self.predict_cov(query, pred_batch_size=pred_batch_size)


class GKRRegressor:
    def __init__(
        self,
        n_input: int,
        n_output: int,
        circular_period: Optional[float | list[Optional[float]]] = None,
        *,
        fit_valid_split: float = 0.3,
        learning_rate: float = 0.1,
        n_epochs: int = 100,
        gpr_params: Optional[dict] = None,
        cov_fit_batch_size: int = 3000,
        kernel_params: Optional[dict] = None,
        dtype: torch.dtype = torch.float64,
        device: Optional[str | torch.device] = None,
        random_state: Optional[int] = 0,
        covariance_neighbors_per_effective_dimension: Optional[float] = None,
        covariance_initialization_grid_size: int = 256,
    ) -> None:
        self.n_input = n_input
        self.n_output = n_output
        self.circular_period = circular_period
        self.fit_valid_split = fit_valid_split
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.cov_fit_batch_size = cov_fit_batch_size
        self.dtype = dtype
        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.random_state = random_state
        self.covariance_neighbors_per_effective_dimension = (
            covariance_neighbors_per_effective_dimension
        )
        self.covariance_initialization_grid_size = int(
            covariance_initialization_grid_size
        )
        self.covariance_initialization_: Optional[dict[str, float]] = None

        gpr_params = dict(gpr_params or {})
        kernel_params = dict(kernel_params or {})

        self.mean_model = IndependentExactGPRegressor(
            n_outputs=n_output,
            circular_period=circular_period,
            dtype=dtype,
            device=self.device,
            **gpr_params,
        )
        self.covariance_model = TorchKernelCovariance(
            n_input=n_input,
            n_output=n_output,
            circular_period=circular_period,
            dtype=dtype,
            device=self.device,
            **kernel_params,
        )

        self.responses_: Optional[torch.Tensor] = None
        self.inputs_: Optional[torch.Tensor] = None
        self.loss_history_: list[float] = []
        self.mean_loss_history_: list[list[float]] = []

        self._generator = torch.Generator(device="cpu")
        if random_state is not None:
            self._generator.manual_seed(int(random_state))

    def _validate_fit_tensors(self, responses: torch.Tensor, inputs: torch.Tensor) -> None:
        if inputs.shape[1] != self.n_input:
            raise ValueError("Input feature dimension does not match n_input.")
        if responses.shape[1] != self.n_output:
            raise ValueError("Response dimension does not match n_output.")

    def _fit_mean_and_residuals(
        self, responses: torch.Tensor, inputs: torch.Tensor
    ) -> torch.Tensor:
        self.mean_model.fit(inputs, responses)
        self.mean_loss_history_ = copy.deepcopy(self.mean_model.mean_loss_history_)
        mean_pred, _ = self.mean_model.predict(inputs)
        return responses - mean_pred

    def _fit_covariance_epoch(
        self,
        residuals: torch.Tensor,
        inputs: torch.Tensor,
        optimizer: torch.optim.Optimizer,
    ) -> float:
        epoch_loss = 0.0
        permutation = torch.randperm(residuals.shape[0], generator=self._generator).to(self.device)

        for start in range(0, residuals.shape[0], self.cov_fit_batch_size):
            batch_idx = permutation[start : start + self.cov_fit_batch_size]
            if batch_idx.numel() == 0:
                continue

            batch_responses = residuals[batch_idx]
            batch_inputs = inputs[batch_idx]
            r_train, r_valid, x_train, x_valid = self._split_train_valid(batch_responses, batch_inputs)

            optimizer.zero_grad()
            self.covariance_model.fit(r_train, x_train)
            cov_pred = self.covariance_model.predict_cov(x_valid)
            loss = -gaussian_log_likelihood(r_valid, cov_pred)
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.detach().cpu())

        return epoch_loss

    def _restore_mean_model_from_payload(self, mean_payload: dict[str, Any]) -> None:
        batch_shape = torch.Size([self.n_output])
        likelihood = gpytorch.likelihoods.GaussianLikelihood(batch_shape=batch_shape).to(
            self.device, self.dtype
        )
        kernel = self.mean_model._kernel_for_batch(self.n_input).to(self.device, self.dtype)

        model_type = str(mean_payload["model_type"])
        if model_type == "batch_exact":
            train_x = torch.as_tensor(mean_payload["train_x"], dtype=self.dtype, device=self.device)
            train_targets = torch.as_tensor(
                mean_payload["train_targets"], dtype=self.dtype, device=self.device
            )
            batch_model: gpytorch.models.GP = _BatchExactGPModel(
                train_x=train_x,
                train_y=train_targets,
                likelihood=likelihood,
                kernel=kernel,
                batch_shape=batch_shape,
            ).to(self.device, self.dtype)
        elif model_type == "batch_sparse":
            inducing_points = torch.as_tensor(
                mean_payload["inducing_points"], dtype=self.dtype, device=self.device
            )
            batch_model = _BatchSparseGPModel(
                inducing_points=inducing_points,
                kernel=kernel,
                batch_shape=batch_shape,
            ).to(self.device, self.dtype)
        else:
            raise ValueError(f"Unsupported mean model type in checkpoint: {model_type}")

        batch_model.load_state_dict(mean_payload["state_dict"])
        likelihood.load_state_dict(mean_payload["likelihood_state_dict"])
        self.mean_model.batch_model_ = batch_model
        self.mean_model.batch_likelihood_ = likelihood
        self.mean_model.models_ = [batch_model]
        self.mean_model.likelihoods_ = [likelihood]
        self.mean_model.output_mean_ = torch.as_tensor(
            mean_payload["output_mean"], dtype=self.dtype, device=self.device
        )
        self.mean_model.output_std_ = torch.as_tensor(
            mean_payload["output_std"], dtype=self.dtype, device=self.device
        )
        self.mean_loss_history_ = copy.deepcopy(mean_payload.get("mean_loss_history", []))
        self.mean_model.mean_loss_history_ = copy.deepcopy(self.mean_loss_history_)

    def _to_serializable_dict(self) -> dict[str, Any]:
        if self.mean_model.batch_model_ is None or self.mean_model.batch_likelihood_ is None:
            raise RuntimeError("Model must be fit before save is called.")
        if self.mean_model.output_mean_ is None or self.mean_model.output_std_ is None:
            raise RuntimeError("Mean-model normalization buffers are missing. Fit before saving.")

        mean_model_obj = self.mean_model.batch_model_
        if isinstance(mean_model_obj, _BatchExactGPModel):
            mean_model_type = "batch_exact"
            extra = {
                "train_x": mean_model_obj.train_inputs[0].detach().cpu(),
                "train_targets": mean_model_obj.train_targets.detach().cpu(),
            }
        elif isinstance(mean_model_obj, _BatchSparseGPModel):
            mean_model_type = "batch_sparse"
            extra = {
                "inducing_points": (
                    mean_model_obj.variational_strategy.inducing_points.detach().cpu()
                )
            }
        else:
            raise TypeError(
                "Unsupported fitted mean model type for serialization: "
                f"{type(mean_model_obj).__name__}"
            )

        payload: dict[str, Any] = {
            "version": 1,
            "init_params": {
                "n_input": self.n_input,
                "n_output": self.n_output,
                "circular_period": self.circular_period,
                "fit_valid_split": self.fit_valid_split,
                "learning_rate": self.learning_rate,
                "n_epochs": self.n_epochs,
                "cov_fit_batch_size": self.cov_fit_batch_size,
                "random_state": self.random_state,
                "covariance_neighbors_per_effective_dimension": (
                    self.covariance_neighbors_per_effective_dimension
                ),
                "covariance_initialization_grid_size": (
                    self.covariance_initialization_grid_size
                ),
                "gpr_params": {
                    "standardize": self.mean_model.standardize,
                    "n_inducing": self.mean_model.n_inducing,
                    "seperate_kernel": self.mean_model.seperate_kernel,
                    "training_iter": self.mean_model.training_iter,
                    "lr": self.mean_model.lr,
                    "verbose": self.mean_model.verbose,
                    "verbose_every": self.mean_model.verbose_every,
                    "track_history": self.mean_model.track_history,
                },
                "kernel_params": {
                    "diag_factor": self.covariance_model.diag_factor,
                },
                "dtype_name": _dtype_to_name(self.dtype),
            },
            "mean_model": {
                "model_type": mean_model_type,
                "state_dict": mean_model_obj.state_dict(),
                "likelihood_state_dict": self.mean_model.batch_likelihood_.state_dict(),
                "output_mean": self.mean_model.output_mean_.detach().cpu(),
                "output_std": self.mean_model.output_std_.detach().cpu(),
                "mean_loss_history": copy.deepcopy(self.mean_loss_history_),
                **extra,
            },
            "covariance_model": {
                "kernel_prec_L": self.covariance_model.kernel_prec_L.detach().cpu(),
                "train_responses": self.covariance_model.train_responses.detach().cpu(),
                "train_inputs": self.covariance_model.train_inputs.detach().cpu(),
            },
            "loss_history": copy.deepcopy(self.loss_history_),
            "covariance_initialization": copy.deepcopy(
                self.covariance_initialization_
            ),
            "responses": None if self.responses_ is None else self.responses_.detach().cpu(),
            "inputs": None if self.inputs_ is None else self.inputs_.detach().cpu(),
        }
        return payload

    def save(self, path: str | Path) -> Path:
        target = Path(path).expanduser()
        target.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self._to_serializable_dict(), target)
        return target

    @classmethod
    def load(
        cls,
        path: str | Path,
        *,
        map_location: Optional[str | torch.device] = "cpu",
        device: Optional[str | torch.device] = None,
    ) -> "GKRRegressor":
        source = Path(path).expanduser()
        payload = torch.load(source, map_location=map_location)
        if int(payload.get("version", -1)) != 1:
            raise ValueError(f"Unsupported checkpoint version: {payload.get('version')}")

        init_params = dict(payload["init_params"])
        dtype = _dtype_from_name(str(init_params.pop("dtype_name")))
        if device is not None:
            init_params["device"] = torch.device(device)
        init_params["dtype"] = dtype
        model = cls(**init_params)

        mean_payload = payload["mean_model"]
        model._restore_mean_model_from_payload(mean_payload)

        cov_payload = payload["covariance_model"]
        with torch.no_grad():
            model.covariance_model.kernel_prec_L.copy_(
                torch.as_tensor(cov_payload["kernel_prec_L"], dtype=model.dtype, device=model.device)
            )
        train_responses = torch.as_tensor(
            cov_payload["train_responses"], dtype=model.dtype, device=model.device
        )
        train_inputs = torch.as_tensor(cov_payload["train_inputs"], dtype=model.dtype, device=model.device)
        if train_responses.numel() > 0 and train_inputs.numel() > 0:
            model.covariance_model.fit(train_responses, train_inputs)
        model.loss_history_ = copy.deepcopy(payload.get("loss_history", []))
        model.covariance_initialization_ = copy.deepcopy(
            payload.get("covariance_initialization")
        )
        model.responses_ = (
            None
            if payload.get("responses") is None
            else torch.as_tensor(payload["responses"], dtype=model.dtype, device=model.device)
        )
        model.inputs_ = (
            None
            if payload.get("inputs") is None
            else torch.as_tensor(payload["inputs"], dtype=model.dtype, device=model.device)
        )
        return model

    def _split_train_valid(self, responses: torch.Tensor, inputs: torch.Tensor) -> tuple[torch.Tensor, ...]:
        batch_size = responses.shape[0]
        if batch_size < 2:
            return responses, responses, inputs, inputs

        n_valid = max(1, int(round(batch_size * self.fit_valid_split)))
        n_valid = min(n_valid, batch_size - 1)
        indices = torch.randperm(batch_size, generator=self._generator)
        valid_idx = indices[:n_valid].to(inputs.device)
        train_idx = indices[n_valid:].to(inputs.device)
        return responses[train_idx], responses[valid_idx], inputs[train_idx], inputs[valid_idx]

    def fit(self, responses: TensorLike, inputs: TensorLike, fit_cov: bool = True) -> list[float]:
        responses_t = _ensure_2d_tensor(responses, dtype=self.dtype, device=self.device)
        inputs_t = _ensure_2d_tensor(inputs, dtype=self.dtype, device=self.device)

        self._validate_fit_tensors(responses_t, inputs_t)

        self.responses_ = responses_t
        self.inputs_ = inputs_t

        residuals = self._fit_mean_and_residuals(responses_t, inputs_t)

        self.covariance_initialization_ = None
        neighbors = self.covariance_neighbors_per_effective_dimension
        if neighbors is not None:
            if not np.isscalar(self.circular_period) or self.n_input != 1:
                raise ValueError(
                    "effective-dimensionality initialization requires one "
                    "periodic scalar input"
                )
            initialization = periodic_covariance_kernel_initialization(
                residuals,
                inputs_t[:, 0],
                period=float(self.circular_period),
                neighbors_per_effective_dimension=float(neighbors),
                grid_size=self.covariance_initialization_grid_size,
            )
            with torch.no_grad():
                self.covariance_model.kernel_prec_L.zero_()
                self.covariance_model.kernel_prec_L[0, 0] = math.sqrt(
                    initialization["initial_precision"]
                )
            self.covariance_initialization_ = initialization

        if not fit_cov:
            self.covariance_model.fit(residuals, inputs_t)
            self.loss_history_ = []
            return self.loss_history_

        optimizer = torch.optim.Adam(self.covariance_model.parameters(), lr=self.learning_rate)
        self.loss_history_ = []

        for _ in range(self.n_epochs):
            epoch_loss = self._fit_covariance_epoch(residuals, inputs_t, optimizer)
            self.loss_history_.append(epoch_loss)

        self.covariance_model.fit(residuals, inputs_t)
        return self.loss_history_

    def predict(
        self,
        query: TensorLike,
        *,
        return_cov: bool = True,
        with_GLASSO: Optional[float] = None,
    ) -> tuple[np.ndarray, Optional[np.ndarray]]:
        query_t = _ensure_2d_tensor(query, dtype=self.dtype, device=self.device)
        mean_pred, _ = self.mean_model.predict(query_t)
        mean_np = mean_pred.detach().cpu().numpy()

        if not return_cov:
            return mean_np, None

        cov_pred = self.covariance_model.predict_cov(query_t).detach().cpu().numpy()
        if with_GLASSO is not None:
            cov_glasso = np.zeros_like(cov_pred)
            for idx, cov_matrix in enumerate(cov_pred):
                glasso = GraphicalLasso(alpha=with_GLASSO, assume_centered=True)
                glasso.fit(cov_matrix)
                cov_glasso[idx] = glasso.covariance_
            cov_pred = cov_glasso

        return mean_np, cov_pred


def circular_ground_truth(
    angles: TensorLike,
    *,
    radius: float = 1.0,
    noise_factor: float = 0.1,
    dtype: torch.dtype = torch.float64,
    device: Optional[str | torch.device] = None,
    as_tensor: bool = False,
) -> tuple[TensorLike, TensorLike]:
    device = torch.device(device or "cpu")
    theta = torch.as_tensor(angles, dtype=dtype, device=device).reshape(-1)
    radial_unit = torch.stack((torch.cos(theta), torch.sin(theta)), dim=1)
    tangent_unit = torch.stack((-torch.sin(theta), torch.cos(theta)), dim=1)
    response_mean = radius * radial_unit

    radial_std = noise_factor * (0.65 + 0.25 * torch.sin(theta))
    tangent_std = noise_factor * (1.00 + 0.35 * torch.cos(2.0 * theta))
    response_cov = (
        radial_std.square()[:, None, None]
        * radial_unit[:, :, None]
        * radial_unit[:, None, :]
        + tangent_std.square()[:, None, None]
        * tangent_unit[:, :, None]
        * tangent_unit[:, None, :]
    )

    if as_tensor:
        return response_mean, response_cov
    return response_mean.cpu().numpy(), response_cov.cpu().numpy()


def generate_circular_dataset(
    n_data: int,
    *,
    radius: float = 1.0,
    noise_factor: float = 0.1,
    dtype: torch.dtype = torch.float64,
    device: Optional[str | torch.device] = None,
    seed: Optional[int] = 0,
    as_tensor: bool = False,
) -> tuple[TensorLike, TensorLike]:
    device = torch.device(device or "cpu")
    generator = torch.Generator(device="cpu")
    if seed is not None:
        generator.manual_seed(int(seed))

    angles = torch.linspace(0.0, 2.0 * math.pi, n_data, dtype=dtype, device=device).unsqueeze(-1)
    response, response_cov = circular_ground_truth(
        angles,
        radius=radius,
        noise_factor=noise_factor,
        dtype=dtype,
        device=device,
        as_tensor=True,
    )
    standard_noise = torch.randn(response.shape, generator=generator, dtype=dtype).to(device)
    noise = torch.einsum("nij,nj->ni", torch.linalg.cholesky(response_cov), standard_noise)
    response_noisy = response + noise

    if as_tensor:
        return response_noisy, angles
    return response_noisy.cpu().numpy(), angles.cpu().numpy()


## Generate toy circular data

In [ ]:
n_data = 100
radius = 1.0
noise_factor = 0.2
response_noisy, labels = generate_circular_dataset(
    n_data,
    radius=radius,
    noise_factor=noise_factor,
    seed=0,
)

plt.figure(figsize=(6, 6))
plt.scatter(response_noisy[:, 0], response_noisy[:, 1], c=labels.ravel(), cmap='viridis')
plt.title('Noisy Circular Data')
plt.xlabel('X')
plt.ylabel('Y')
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

## Fit GKR with the updated implementation

The interface stays close to the original demo: create `GKRRegressor`, call `fit`, then `predict`.

In [ ]:
gkr = GKRRegressor(
    n_input=labels.shape[1],
    n_output=response_noisy.shape[1],
    circular_period=2 * np.pi,
    n_epochs=100,
    cov_fit_batch_size=64,
    random_state=0,
    covariance_neighbors_per_effective_dimension=10.0,
    covariance_initialization_grid_size=256,
    gpr_params={
        'training_iter': 75,
        'lr': 0.12,
        'n_inducing': 20,
        'verbose': True,
        'verbose_every': 25,
        'track_history': True,
    },
)

loss_history = gkr.fit(response_noisy, labels, fit_cov=True)
print('Covariance-kernel initialization:', gkr.covariance_initialization_)
print(f'Training finished. Epoch losses: {loss_history}')

## Predict manifold and local covariance

For this toy generator, the ground-truth conditional mean is $(r\cos\theta, r\sin\theta)$. Its covariance varies smoothly with angle in the local radial and tangential directions: $\sigma_r(\theta)=\sigma(0.65+0.25\sin\theta)$ and $\sigma_t(\theta)=\sigma(1+0.35\cos 2\theta)$. The plot evaluates both truth and GKR at the same conditions.

In [ ]:
label_pred = np.linspace(0, 2 * np.pi, 100).reshape(-1, 1)
response_pred, response_cov = gkr.predict(label_pred)
response_true, response_cov_true = circular_ground_truth(
    label_pred,
    radius=radius,
    noise_factor=noise_factor,
)

print('Predicted mean shape:', response_pred.shape)
print('Predicted covariance shape:', response_cov.shape)
print('All finite:', np.isfinite(response_pred).all() and np.isfinite(response_cov).all())

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6))
ax.scatter(
    response_noisy[:, 0],
    response_noisy[:, 1],
    c=labels.ravel(),
    cmap='viridis',
    alpha=0.55,
    label='Noisy samples',
    zorder=2,
)
ax.plot(
    response_true[:, 0],
    response_true[:, 1],
    color='#2f2f2f',
    linestyle='--',
    linewidth=2,
    label='Ground-truth mean',
    zorder=3,
)
ax.plot(
    response_pred[:, 0],
    response_pred[:, 1],
    color='#c23b2a',
    linewidth=2,
    label='GKR mean',
    zorder=4,
)

for idx in range(0, len(response_pred), 8):
    plot_cov_ellipse(
        response_pred[idx],
        response_cov[idx],
        ax,
        alpha=0.16,
        color='#c23b2a',
        label='GKR covariance' if idx == 0 else '_nolegend_',
        zorder=1,
    )
    plot_cov_ellipse(
        response_true[idx],
        response_cov_true[idx],
        ax,
        facecolor='none',
        edgecolor='#2f2f2f',
        linestyle='--',
        linewidth=1.4,
        label='Ground-truth covariance' if idx == 0 else '_nolegend_',
        zorder=5,
    )

ax.set_title('GKR Manifold and Local Covariance')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_aspect('equal', adjustable='box')
ax.legend(
    frameon=False,
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
)
fig.tight_layout()
plt.show()

## Optional: save outputs and reload the fitted model

In [ ]:
np.savez(
    'gkr_demo_outputs.npz',
    response_noisy=response_noisy,
    labels=labels,
    response_pred=response_pred,
    response_cov=response_cov,
)

checkpoint_path = gkr.save('gkr_demo_checkpoint.pt')
gkr_reloaded = GKRRegressor.load(checkpoint_path)
response_pred_reload, response_cov_reload = gkr_reloaded.predict(label_pred)

print('Saved outputs to gkr_demo_outputs.npz')
print('Saved checkpoint to', checkpoint_path)
print('Reloaded mean match:', np.allclose(response_pred, response_pred_reload, atol=1e-6))
print('Reloaded cov match:', np.allclose(response_cov, response_cov_reload, atol=1e-6))